# Inference Performance Table
Reads `*_perf.yaml` files from each PDE's performance directory,
computes derived metrics, and outputs the LaTeX table.

In [1]:
# ── Cell 1 — Imports ──────────────────────────────────────────────────────────
import yaml
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# ── Cell 2 — Config: edit these paths / patterns ──────────────────────────────

PROJECT_ROOT = Path("/Users/gabrieleaccarino/Workspace/waveletflux")

PDES = [
    "turbulent_radiative_layer_2D",
    "rayleigh_benard",
    "active_matter",
]

# Glob prefixes to discover YAML files inside each performance/directory.
# A file matches if its name starts with one of these strings.
MODEL_PATTERNS = [
    "FM_data_space",
    "WaveletFlow_3scales_haar_v4.0",
    "WaveletFlow_3scales_db2_v4.0",
    "WaveletFlow_3scales_db4_v4.0",
    "WaveletFlow_3scales_db6_v4.0",
]

# NFE used during inference (same for all models in this benchmark)
NFE = 50

In [3]:
# ── Cell 3 — YAML loading helpers ────────────────────────────────────────────

def avg_gpu_mem(block):
    """Return mean peak GPU memory across all ranks (GiB)."""
    if block is None:
        return None
    if isinstance(block, (int, float)):
        return float(block)
    if isinstance(block, dict):
        vals = [float(v) for v in block.values() if v is not None]
        return float(np.mean(vals)) if vals else None
    return None


def resolve_model_name(filename: str, pde: str) -> str:
    """Strip the known _{pde}_perf.yaml suffix to recover the model name."""
    for suffix in (f"_{pde}_perf.yaml", f"_{pde}_performance.yaml"):
        if filename.endswith(suffix):
            return filename[: -len(suffix)]
    return Path(filename).stem


def load_yaml(path: Path) -> dict:
    with open(path) as f:
        raw = yaml.safe_load(f)
    wc = raw.get("wall_clock_s")
    ts = raw.get("total_steps_global")
    return {
        "wall_clock_s":        wc,
        "total_steps_global":  ts,
        "gpu_peak_mem_GiB":    avg_gpu_mem(raw.get("gpu_peak_mem_GiB")),
        "throughput_steps_s":  raw.get("throughput_steps_s"),
        "throughput_frames_s": raw.get("throughput_frames_s"),
        # derived
        "sec_per_rollout":     round(wc / ts, 6) if (wc and ts) else None,
    }

In [4]:
# ── Cell 4 — Discover and load all YAML files ─────────────────────────────────

records = []

for pde in PDES:
    perf_dir = PROJECT_ROOT / "experiments" / pde / "performance"
    if not perf_dir.exists():
        print(f"[MISS] {perf_dir}")
        continue

    found: dict[str, Path] = {}
    for pattern in MODEL_PATTERNS:
        for p in sorted(perf_dir.glob(f"{pattern}*_perf.yaml")) + \
                 sorted(perf_dir.glob(f"{pattern}*_performance.yaml")):
            name = resolve_model_name(p.name, pde)
            found.setdefault(name, p)      # first match wins

    if not found:
        print(f"[MISS] no matching YAMLs in {perf_dir}")
        continue

    for model_name, yaml_path in sorted(found.items()):
        print(f"  ✓  {yaml_path.name}")
        try:
            m = load_yaml(yaml_path)
            records.append({"pde": pde, "model": model_name, **m})
        except Exception as e:
            print(f"    [ERROR] {yaml_path}: {e}")

df = pd.DataFrame(records)
df["throughput_Mframes_s"] = df["throughput_frames_s"] / 1e6
print(f"\nLoaded {len(df)} rows")
df

  ✓  FM_data_space_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  FM_data_space_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_3scales_db4_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_3scales_db6_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_3scales_haar_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  FM_data_space_v4.0_nsampling50_nens8_active_matter_perf.yaml
  ✓  WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_active_matter_perf.yaml
  ✓  WaveletFlow_3scales_db4_v4

,pde,model,wall_clock_s,total_steps_global,gpu_peak_mem_GiB,throughput_steps_s,throughput_frames_s,sec_per_rollout,throughput_Mframes_s
0,turbulent_radiative_layer_2D,FM_data_space_v4.0_nsampling50_nens8,247.75,6984,0.590,28.2,1385580.0,0.035474,1.385580
1,turbulent_radiative_layer_2D,WaveletFlow_3scales_db2_v4.0_nsampling50_nens8,224.02,6984,0.663,31.2,1532380.0,0.032076,1.532380
2,turbulent_radiative_layer_2D,WaveletFlow_3scales_db4_v4.0_nsampling50_nens8,228.23,6984,0.663,30.6,1504072.0,0.032679,1.504072
3,turbulent_radiative_layer_2D,WaveletFlow_3scales_db6_v4.0_nsampling50_nens8,221.05,6984,0.663,31.6,1552952.8,0.031651,1.552953
4,turbulent_radiative_layer_2D,WaveletFlow_3scales_haar_v4.0_nsampling50_nens8,218.27,6984,0.663,32.0,1572733.1,0.031253,1.572733
5,rayleigh_benard,FM_data_space_v4.0_nsampling50_nens8,633.85,14112,1.441,22.3,1459093.4,0.044916,1.459093
6,rayleigh_benard,WaveletFlow_3scales_db2_v4.0_nsampling50_nens8,446.53,14112,1.514,31.6,2071182.7,0.031642,2.071183
7,rayleigh_benard,WaveletFlow_3scales_db4_v4.0_nsampling50_nens8,457.89,14112,1.515,30.8,2019777.5,0.032447,2.019778
8,rayleigh_benard,WaveletFlow_3scales_db6_v4.0_nsampling50_nens8,467.97,14112,1.515,30.2,1976275.2,0.033161,1.976275
9,rayleigh_benard,WaveletFlow_3scales_haar_v4.0_nsampling50_nens8,452.97,14112,1.513,31.2,2041721.0,0.032098,2.041721


In [5]:
# ── Cell 5 — Classify models and compute WFM-vs-FM ratios ────────────────────

WFM_KEYWORDS = ["WaveletFlow", "WFM", "Wavelet"]

def is_wfm(name: str) -> bool:
    return any(k.lower() in name.lower() for k in WFM_KEYWORDS)

df["model_type"] = df["model"].apply(lambda m: "WFM" if is_wfm(m) else "FM")

# Ratio metrics: direction="lower" → FM/WFM  |  "higher" → WFM/FM
RATIO_METRICS = [
    ("wall_clock_s",        "lower"),
    ("sec_per_rollout",     "lower"),
    ("gpu_peak_mem_GiB",    "lower"),
    ("throughput_Mframes_s","higher"),
]

for m, _ in RATIO_METRICS:
    df[f"ratio_{m}"] = np.nan

for pde, grp in df.groupby("pde"):
    fm = grp[grp["model_type"] == "FM"]
    if fm.empty:
        print(f"[WARN] no FM baseline for {pde}")
        continue
    fm_row = fm.iloc[0]
    for idx in grp[grp["model_type"] == "WFM"].index:
        for metric, direction in RATIO_METRICS:
            fv = fm_row.get(metric)
            wv = df.at[idx, metric]
            if pd.notna(fv) and pd.notna(wv) and fv != 0:
                df.at[idx, f"ratio_{metric}"] = (
                    wv / fv if direction == "higher" else fv / wv
                )

df[["pde","model","model_type",
    "wall_clock_s","sec_per_rollout","gpu_peak_mem_GiB","throughput_Mframes_s",
    "ratio_wall_clock_s","ratio_gpu_peak_mem_GiB"]]

,pde,model,model_type,wall_clock_s,sec_per_rollout,gpu_peak_mem_GiB,throughput_Mframes_s,ratio_wall_clock_s,ratio_gpu_peak_mem_GiB
0,turbulent_radiative_layer_2D,FM_data_space_v4.0_nsampling50_nens8,FM,247.75,0.035474,0.590,1.385580,NaN,NaN
1,turbulent_radiative_layer_2D,WaveletFlow_3scales_db2_v4.0_nsampling50_nens8,WFM,224.02,0.032076,0.663,1.532380,1.105928,0.889894
2,turbulent_radiative_layer_2D,WaveletFlow_3scales_db4_v4.0_nsampling50_nens8,WFM,228.23,0.032679,0.663,1.504072,1.085528,0.889894
3,turbulent_radiative_layer_2D,WaveletFlow_3scales_db6_v4.0_nsampling50_nens8,WFM,221.05,0.031651,0.663,1.552953,1.120787,0.889894
4,turbulent_radiative_layer_2D,WaveletFlow_3scales_haar_v4.0_nsampling50_nens8,WFM,218.27,0.031253,0.663,1.572733,1.135062,0.889894
5,rayleigh_benard,FM_data_space_v4.0_nsampling50_nens8,FM,633.85,0.044916,1.441,1.459093,NaN,NaN
6,rayleigh_benard,WaveletFlow_3scales_db2_v4.0_nsampling50_nens8,WFM,446.53,0.031642,1.514,2.071183,1.419501,0.951783
7,rayleigh_benard,WaveletFlow_3scales_db4_v4.0_nsampling50_nens8,WFM,457.89,0.032447,1.515,2.019778,1.384284,0.951155
8,rayleigh_benard,WaveletFlow_3scales_db6_v4.0_nsampling50_nens8,WFM,467.97,0.033161,1.515,1.976275,1.354467,0.951155
9,rayleigh_benard,WaveletFlow_3scales_haar_v4.0_nsampling50_nens8,WFM,452.97,0.032098,1.513,2.041721,1.399320,0.952412


In [8]:
# ── Cell 6 — Label maps and display pivot ─────────────────────────────────────

MODEL_LABELS = {
    "FM_data_space":      r"$\mathrm{FM}_{\mathrm{pixel}}$",
    "WaveletFlow_3scales_haar": "WFM 3-scales haar",
    "WaveletFlow_3scales_db2": "WFM 3-scales db2",
    "WaveletFlow_3scales_db4": "WFM 3-scales db4",
    "WaveletFlow_3scales_db6": "WFM 3-scales db6",
}

def shorten_model(name: str) -> str:
    for key, label in MODEL_LABELS.items():
        if key in name:
            return label
    return name

MODEL_ORDER = list(MODEL_LABELS.values())

PDE_LABELS = {
    "turbulent_radiative_layer_2D": "Turb. Rad. Layer 2D",
    "rayleigh_benard":              "Rayleigh--Bénard",
    "active_matter":                "Active Matter",
}
PDE_ORDER = list(PDE_LABELS.values())

df2 = df.copy()
df2["model_short"] = df2["model"].apply(shorten_model)
df2["pde_label"]   = df2["pde"].map(PDE_LABELS)

# (df_column, display_name, format_spec, show_for_FM)
METRICS = [
    ("wall_clock_s",        "Wall Clock (s)",     ".1f",  True),
    ("sec_per_rollout",     "sec / step",         ".4f",  True),
    ("gpu_peak_mem_GiB",    "GPU Mem (GiB)",      ".2f",  True),
    ("throughput_Mframes_s","Throughput (Mfr/s)", ".2f",  True),
    ("ratio_wall_clock_s",  "Speedup ×",          ".2f",  False),
    ("ratio_gpu_peak_mem_GiB","Mem ratio ×",      ".2f",  False),
]
METRIC_COLS = [label for _, label, _, _ in METRICS]
fmt_map     = {label: fmt  for _, label, fmt, _    in METRICS}

rows, index = [], []
for pde_label in PDE_ORDER:
    for model in MODEL_ORDER:
        is_fm = "FM" in model and "WFM" not in model
        cell  = df2[(df2["pde_label"] == pde_label) & (df2["model_short"] == model)]
        row   = {}
        for col, label, fmt, show_for_fm in METRICS:
            if cell.empty or col not in cell.columns:
                row[label] = np.nan
            elif is_fm and not show_for_fm:
                row[label] = np.nan
            else:
                row[label] = cell[col].values[0]
        rows.append(row)
        index.append((pde_label, model))

pivot = pd.DataFrame(rows, columns=METRIC_COLS,
                     index=pd.MultiIndex.from_tuples(index, names=["PDE", "Model"]))

def fmt_cell(val, metric_name):
    if pd.isna(val):
        return "—"
    result = f"{val:{fmt_map.get(metric_name, '.3f')}}"
    if "×" in metric_name:
        result += "×"
    return result

pivot_fmt = pivot.copy().astype(object)
for metric in METRIC_COLS:
    pivot_fmt[metric] = pivot[metric].apply(lambda v: fmt_cell(v, metric))

pivot_fmt

Wall Clock (s) sec / step  \
PDE                 Model                                                      
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$          247.8     0.0355   
                    WFM 3-scales haar                       218.3     0.0313   
                    WFM 3-scales db2                        224.0     0.0321   
                    WFM 3-scales db4                        228.2     0.0327   
                    WFM 3-scales db6                        221.1     0.0317   
Rayleigh--Bénard    $\mathrm{FM}_{\mathrm{pixel}}$          633.9     0.0449   
                    WFM 3-scales haar                       453.0     0.0321   
                    WFM 3-scales db2                        446.5     0.0316   
                    WFM 3-scales db4                        457.9     0.0324   
                    WFM 3-scales db6                        468.0     0.0332   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$          310.8     0.0459   
                    WFM 3-scales haar                       244.7     0.0361   
                    WFM 3-scales db2                        247.4     0.0365   
                    WFM 3-scales db4                        252.6     0.0373   
                    WFM 3-scales db6                        261.2     0.0386   

                                                   GPU Mem (GiB)  \
PDE                 Model                                          
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$          0.59   
                    WFM 3-scales haar                       0.66   
                    WFM 3-scales db2                        0.66   
                    WFM 3-scales db4                        0.66   
                    WFM 3-scales db6                        0.66   
Rayleigh--Bénard    $\mathrm{FM}_{\mathrm{pixel}}$          1.44   
                    WFM 3-scales haar                       1.51   
                    WFM 3-scales db2                        1.51   
                    WFM 3-scales db4                        1.51   
                    WFM 3-scales db6                        1.51   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$          1.63   
                    WFM 3-scales haar                       1.71   
                    WFM 3-scales db2                        1.71   
                    WFM 3-scales db4                        1.71   
                    WFM 3-scales db6                        1.71   

                                                   Throughput (Mfr/s)  \
PDE                 Model                                               
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$               1.39   
                    WFM 3-scales haar                            1.57   
                    WFM 3-scales db2                             1.53   
                    WFM 3-scales db4                             1.50   
                    WFM 3-scales db6                             1.55   
Rayleigh--Bénard    $\mathrm{FM}_{\mathrm{pixel}}$               1.46   
                    WFM 3-scales haar                            2.04   
                    WFM 3-scales db2                             2.07   
                    WFM 3-scales db4                             2.02   
                    WFM 3-scales db6                             1.98   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$               1.43   
                    WFM 3-scales haar                            1.82   
                    WFM 3-scales db2                             1.80   
                    WFM 3-scales db4                             1.76   
                    WFM 3-scales db6                             1.70   

                                                   Speedup × Mem ratio ×  
PDE                 Model                                                 
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$         —           —  
                    WFM 3-scales haar                  1.14×      

In [9]:
# ── Cell 7 — LaTeX table ──────────────────────────────────────────────────────

PDE_LATEX = {
    "Turb. Rad. Layer 2D": "Turbulence Radiative Layer 2D",
    "Rayleigh-Bénard":     "Rayleigh-B\\'{e}nard",
    "Active Matter":       "Active Matter",
}

MODEL_LATEX = {
    r"$\mathrm{FM}_{\mathrm{pixel}}$": r"$FM_{pixel}$",
    "WFM 1-scale":  "WFM 1-scale",
    "WFM 3-scales": "WFM 3-scales",
}

caption = (
    r"Inference performance across PDEs and models. "
    r"\textit{Wall Clock}: total inference time (max across GPU ranks). "
    r"\textit{sec/step}: wall-clock time per autoregressive rollout step "
    r"(\texttt{wall\_clock\_s} $/ $ \texttt{total\_steps\_global}), "
    r"resolution-independent. "
    r"\textit{GPU Mem}: peak GPU memory averaged across ranks. "
    r"\textit{Throughput}: millions of spatial frames per second "
    r"(\texttt{total\_steps} $\times H \times W$ / \texttt{wall\_clock\_s}). "
    r"\textit{Speedup}: wall-clock ratio FM\,/\,WFM; ${>}1$ means WFM is faster. "
    r"\textit{Mem ratio}: peak GPU memory ratio FM\,/\,WFM; ${>}1$ means WFM uses less memory. "
    r"All runs use NFE\,=\,50 ODE steps, 4 GPUs, and 8 ensemble members."
)

lines = []
lines.append(r"\begin{table}[ht]")
lines.append(r"\caption{" + caption + "}")
lines.append(r"\label{tab:inference_perf}")
lines.append(r"\resizebox{\textwidth}{!}{%")
lines.append(r"\begin{tabular}{ll rrrr rr}")
lines.append(r"\toprule")
lines.append(
    r"\textbf{PDE} & \textbf{Model} "
    r"& \textbf{Wall Clock (s)} "
    r"& \textbf{sec / step} "
    r"& \textbf{GPU Mem (GiB)} "
    r"& \textbf{Throughput (Mfr/s)} "
    r"& \textbf{Speedup $\times$} "
    r"& \textbf{Mem ratio $\times$} \\\\"
)

for pde_label in PDE_ORDER:
    lines.append(r"\midrule")
    pde_latex = PDE_LATEX.get(pde_label, pde_label)
    models_in_pde = MODEL_ORDER
    lines.append(rf"\multirow{{{len(models_in_pde)}}}{{*}}{{{pde_latex}}}")

    for i, model in enumerate(models_in_pde):
        model_latex = MODEL_LATEX.get(model, model)
        row_data    = pivot_fmt.loc[(pde_label, model)]
        vals        = " & ".join(
            str(row_data[m]).replace("×", r"$\times$") for m in METRIC_COLS
        )
        # replace — with proper LaTeX dash
        vals = vals.replace("—", r"\textemdash")
        pde_col = "" if i > 0 else ""   # multirow handles the PDE cell
        lines.append(f"  & {model_latex} & {vals} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}%")
lines.append("}")
lines.append(r"\end{table}")

latex = "\n".join(lines)
print(latex)

\begin{table}[ht]
\caption{Inference performance across PDEs and models. \textit{Wall Clock}: total inference time (max across GPU ranks). \textit{sec/step}: wall-clock time per autoregressive rollout step (\texttt{wall\_clock\_s} $/ $ \texttt{total\_steps\_global}), resolution-independent. \textit{GPU Mem}: peak GPU memory averaged across ranks. \textit{Throughput}: millions of spatial frames per second (\texttt{total\_steps} $\times H \times W$ / \texttt{wall\_clock\_s}). \textit{Speedup}: wall-clock ratio FM\,/\,WFM; ${>}1$ means WFM is faster. \textit{Mem ratio}: peak GPU memory ratio FM\,/\,WFM; ${>}1$ means WFM uses less memory. All runs use NFE\,=\,50 ODE steps, 4 GPUs, and 8 ensemble members.}
\label{tab:inference_perf}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll rrrr rr}
\toprule
\textbf{PDE} & \textbf{Model} & \textbf{Wall Clock (s)} & \textbf{sec / step} & \textbf{GPU Mem (GiB)} & \textbf{Throughput (Mfr/s)} & \textbf{Speedup $\times$} & \textbf{Mem ratio $\times$} \\\\
\

In [8]:
# ── Cell 8 — Save to .tex file ────────────────────────────────────────────────
#out = Path("performance_table.tex")
#out.write_text(latex)
#print(f"Saved → {out.resolve()}")